In [ ]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Internship project/dataset"

In [ ]:
persons = sorted([
    folder for folder in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, folder))
])

print(f"Total Personas : {len(persons)}")

for person in persons:
    print(person)

Total Personas : 11
ARAVIND MENON
Abdul Rahman chats
Abhijith +2
Alan techie
Arjun Toyota
Maria Nurse
Priya Nair chats
ROHAN MENON
Suresh Auto
adithya chats
farzan chats


In [ ]:
MESSAGE_PATTERN = re.compile(
    r"^(\d{2}/\d{2}/\d{2},\s\d{1,2}:\d{2}\s(?:AM|PM))\s-\s([^:]+):\s?(.*)"
)

In [ ]:
SYSTEM_EVENTS = [

    "<Media omitted>",
    "<Photo omitted>",
    "<Video omitted>",
    "<Sticker omitted>",
    "<GIF omitted>",
    "<Document omitted>",

    "Voice call",
    "Video call",
    "Missed voice call",
    "Missed video call",

    "Live location shared",
    "Location shared",

    "This message was deleted",
    "You deleted this message"
]

In [ ]:
SEPARATOR_PATTERN = re.compile(r"^[─—-]{5,}$")

In [ ]:
def parse_whatsapp_chat(file_path, person, chat_name):

    messages = []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    current_message = None

    for line in lines:

        line = line.strip()

        # ---------------------------------------
        # Skip empty lines
        # ---------------------------------------
        if line == "":
            continue

        # ---------------------------------------
        # Skip separators
        # ---------------------------------------
        if SEPARATOR_PATTERN.match(line):
            continue

        # ---------------------------------------
        # Check for new WhatsApp message
        # ---------------------------------------
        match = MESSAGE_PATTERN.match(line)

        if match:

            # Save previous message
            if current_message is not None:
                messages.append(current_message)

            timestamp = match.group(1)
            sender = match.group(2).strip()
            message = match.group(3).strip()

            current_message = {
                "Person": person,
                "Chat_Name": chat_name,
                "Timestamp": timestamp,
                "Sender": sender,
                "Message": message
            }

        else:

            # Continuation of previous message
            if current_message is not None:

                current_message["Message"] += "\n" + line

    # Save last message
    if current_message is not None:
        messages.append(current_message)

    return messages

In [ ]:
all_messages = []

for person in persons:

    person_folder = os.path.join(DATASET_PATH, person)

    txt_files = sorted(Path(person_folder).glob("*.txt"))

    print(f"{person} : {len(txt_files)} chats")

    for txt_file in txt_files:

        chat_name = txt_file.stem

        parsed = parse_whatsapp_chat(
            txt_file,
            person,
            chat_name
        )

        all_messages.extend(parsed)

ARAVIND MENON : 22 chats
Abdul Rahman chats : 22 chats
Abhijith +2 : 40 chats
Alan techie : 29 chats
Arjun Toyota : 47 chats
Maria Nurse : 40 chats
Priya Nair chats : 21 chats
ROHAN MENON : 17 chats
Suresh Auto : 34 chats
adithya chats : 37 chats
farzan chats : 29 chats


In [ ]:
df = pd.DataFrame(all_messages)

print(df.shape)

df.head()

(19936, 5)


,Person,Chat_Name,Timestamp,Sender,Message
0,ARAVIND MENON,Chat wiith Coach Manoj,"04/01/26, 07:00 AM",Coach Manoj,"\nMorning! New year, new goals right? What are..."
1,ARAVIND MENON,Chat wiith Coach Manoj,"04/01/26, 08:30 AM",You,\nHonestly just consistency. Roster makes it h...
2,ARAVIND MENON,Chat wiith Coach Manoj,"04/01/26, 08:32 AM",Coach Manoj,\nFair enough. Let's do a flexible split then ...
3,ARAVIND MENON,Chat wiith Coach Manoj,"04/01/26, 08:35 AM",You,\nThat works. When can we start?
4,ARAVIND MENON,Chat wiith Coach Manoj,"04/01/26, 08:36 AM",Coach Manoj,\nTomorrow morning if you're free. 6 AM slot o...


In [ ]:
print(df.isnull().sum())

Person       0
Chat_Name    0
Timestamp    0
Sender       0
Message      0
dtype: int64


In [ ]:
print("Empty Messages:", (df["Message"] == "").sum())

Empty Messages: 0


In [ ]:
df[df["Message"].str.contains("────", na=False)]

,Person,Chat_Name,Timestamp,Sender,Message


In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 104


In [ ]:
df.sample(10, random_state=42)

,Person,Chat_Name,Timestamp,Sender,Message
803,ARAVIND MENON,Chat with Meera S,"18/02/26, 09:30 PM",Meera,\n📸 <Screenshot omitted>
1350,Abdul Rahman chats,Basheer,"27/01/26, 09:05 PM",Basheer,\nഈ നമ്പർ സേവ് ചെയ്യണ്ട.
322,ARAVIND MENON,Chat with Aparna,"19/03/26, 09:06 PM",You,\n📷 <Media omitted>
17486,adithya chats,Chat with chechi,"15/02/26, 07:44 PM",Chechi 👩,"Candle, flowers, chocolate... safe combo 😂"
10963,Priya Nair chats,class teacher,"27/03/26, 12:52 PM",Priya,\nThank you so much miss.\nWe will continue pr...
16559,adithya chats,Chat with Kutty ❤️,"29/05/26, 11:26 AM",Kutty ❤️,Fridayyy 😌
10826,Priya Nair chats,Husband,"30/06/26, 01:22 PM",Arun Ettan 🇦🇪,\nSalary kittumbo ayakkam ❤️
10008,Priya Nair chats,Husband,"04/03/26, 06:36 PM",Priya,\n📷 <Media omitted>\nEvening walk 😄
13320,adithya chats,Auto chettan,"09/01/26, 08:27 AM",Auto Chettan 🚖,\nPurath ethi.
8174,Maria Nurse,chat with athira,"28/04/26, 07:45 PM",You,\nAa.\n6:15 bus pidikkanam.


In [ ]:
duplicates = df[
    df.duplicated(
        subset=["Person","Chat_Name","Timestamp","Sender","Message"],
        keep=False
    )
].sort_values(
    ["Person","Chat_Name","Timestamp"]
)

duplicates

,Person,Chat_Name,Timestamp,Sender,Message
2772,Abhijith +2,football boys,"09/04/26, 08:12 PM",Arun,\nGuys\nSunday 6:30 AM okay?
2777,Abhijith +2,football boys,"09/04/26, 08:12 PM",Arun,\nGuys\nSunday 6:30 AM okay?
2773,Abhijith +2,football boys,"09/04/26, 08:13 PM",Nikhil,\nOkay
2778,Abhijith +2,football boys,"09/04/26, 08:13 PM",Nikhil,\nOkay
2774,Abhijith +2,football boys,"09/04/26, 08:14 PM",You,\nNjan varam
...,...,...,...,...,...
13192,Suresh Auto,chat with wife,"06/03/26, 07:12 PM",Meera ❤️,Toy shopil ninn gift eduthalo?
13179,Suresh Auto,chat with wife,"06/03/26, 07:19 PM",You,Aa... pillerk ishtam aavunna enthelum nokkam.
13193,Suresh Auto,chat with wife,"06/03/26, 07:19 PM",You,Aa... pillerk ishtam aavunna enthelum nokkam.
13180,Suresh Auto,chat with wife,"06/03/26, 07:20 PM",Meera ❤️,👍


In [ ]:
print(len(duplicates))

208


In [ ]:
duplicates.head(20)

,Person,Chat_Name,Timestamp,Sender,Message
2772,Abhijith +2,football boys,"09/04/26, 08:12 PM",Arun,\nGuys\nSunday 6:30 AM okay?
2777,Abhijith +2,football boys,"09/04/26, 08:12 PM",Arun,\nGuys\nSunday 6:30 AM okay?
2773,Abhijith +2,football boys,"09/04/26, 08:13 PM",Nikhil,\nOkay
2778,Abhijith +2,football boys,"09/04/26, 08:13 PM",Nikhil,\nOkay
2774,Abhijith +2,football boys,"09/04/26, 08:14 PM",You,\nNjan varam
2779,Abhijith +2,football boys,"09/04/26, 08:14 PM",You,\nNjan varam
2775,Abhijith +2,football boys,"09/04/26, 08:15 PM",Rahul,\nLate aavaruth 😂
2780,Abhijith +2,football boys,"09/04/26, 08:15 PM",Rahul,\nLate aavaruth 😂
2776,Abhijith +2,football boys,"09/04/26, 08:16 PM",You,\n😂
2781,Abhijith +2,football boys,"09/04/26, 08:16 PM",You,\n😂


In [ ]:
print(df.shape)

(19936, 5)


In [ ]:
duplicates[
    ["Person", "Chat_Name", "Timestamp", "Sender", "Message"]
].to_string()

'             Person                            Chat_Name           Timestamp     Sender                                                                 Message\n2772    Abhijith +2                        football boys  09/04/26, 08:12 PM       Arun                                            \\nGuys\\nSunday 6:30 AM okay?\n2777    Abhijith +2                        football boys  09/04/26, 08:12 PM       Arun                                            \\nGuys\\nSunday 6:30 AM okay?\n2773    Abhijith +2                        football boys  09/04/26, 08:13 PM     Nikhil                                                                  \\nOkay\n2778    Abhijith +2                        football boys  09/04/26, 08:13 PM     Nikhil                                                                  \\nOkay\n2774    Abhijith +2                        football boys  09/04/26, 08:14 PM        You                                                            \\nNjan varam\n2779    Abhijith +2       

In [ ]:
system_events = df[
    df["Sender"].str.contains(
        "Voice|Video|Missed|Messages and calls",
        case=False,
        na=False
    )
]

print(system_events.shape)

system_events.head(20)

(0, 5)


,Person,Chat_Name,Timestamp,Sender,Message


In [ ]:
print("Before:", len(df))

df = df.drop_duplicates(
    subset=[
        "Person",
        "Chat_Name",
        "Timestamp",
        "Sender",
        "Message"
    ],
    keep="first"
).reset_index(drop=True)

print("After:", len(df))

Before: 19936
After: 19832


In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [ ]:
df.to_csv(
    "/content/drive/MyDrive/Internship project/master_whatsapp_dataset.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!
